# NCAA 2026 Raddar 流水线

完整复刻 vilnius-ncaa.ipynb 核心逻辑，生成 submission_raddar.csv。

**环境说明**：若遇到 `ImportError: cannot import name '_lazywhere'`，请升级 statsmodels：
```bash
pip install --upgrade statsmodels
```
或使用 Kaggle 环境运行。

In [2]:
# NCAA 2026 Raddar 主流程
# 完整复刻 vilnius-ncaa.ipynb 核心逻辑
# 数据目录、输出路径见下方

import sys

sys.path.insert(0, "..")

from src.raddar_pipeline import run_raddar

# 运行流水线，生成 submission_raddar.csv
out = run_raddar(
    data_dir="../march-machine-learning-mania-2026",
    output_path="../submission_raddar.csv",
)
print(f"生成完成: {len(out)} 行")
print(out.head(10))

XGB LOO: 100%|██████████| 23/23 [00:38<00:00,  1.68s/it]


生成完成: 132133 行
               ID      Pred
0  2026_1101_1102  0.546294
1  2026_1101_1103  0.302930
2  2026_1101_1104  0.254738
3  2026_1101_1105  0.510044
4  2026_1101_1106  0.514015
5  2026_1101_1107  0.392993
6  2026_1101_1108  0.470763
7  2026_1101_1110  0.400994
8  2026_1101_1111  0.437340
9  2026_1101_1112  0.147544


## 历史测试：LOSO vs 单模型(无 LOSO)

对比两种方式的 Brier 分数：
- **LOSO**：每赛季用「排除该赛季」的模型 OOF 预测，spline 校准
- **单模型(无 LOSO)**：每赛季用「仅过去赛季」训练的模型预测，无未来泄露

In [ ]:
# 运行历史测试，对比有/无 LOSO 的 Brier 分数
import sys

sys.path.insert(0, "..")

from src.raddar_pipeline import run_historical_comparison

result = run_historical_comparison(data_dir="../march-machine-learning-mania-2026")
# result 含: loso_brier, single_brier, loso_per_season, single_per_season

单模型(过去赛季): 100%|██████████| 23/23 [00:33<00:00,  1.44s/it]


========== 历史测试：LOSO vs 单模型(过去赛季) ==========
LOSO OOF Brier (整体):     0.16568051
单模型(过去赛季) Brier:   0.16880667

按赛季对比 (LOSO / 单模型):
  2003: 0.18088 /   -   
  2004: 0.16909 / 0.20930  (Δ=+0.04021)
  2005: 0.16620 / 0.19941  (Δ=+0.03320)
  2006: 0.19274 / 0.20644  (Δ=+0.01371)
  2007: 0.14037 / 0.16184  (Δ=+0.02147)
  2008: 0.15277 / 0.15920  (Δ=+0.00643)
  2009: 0.16593 / 0.17096  (Δ=+0.00503)
  2010: 0.16839 / 0.16951  (Δ=+0.00112)
  2011: 0.18265 / 0.18209  (Δ=-0.00056)
  2012: 0.15333 / 0.15684  (Δ=+0.00351)
  2013: 0.17145 / 0.18603  (Δ=+0.01458)
  2014: 0.17597 / 0.16950  (Δ=-0.00646)
  2015: 0.14261 / 0.14377  (Δ=+0.00116)
  2016: 0.17439 / 0.17616  (Δ=+0.00177)
  2017: 0.16039 / 0.16073  (Δ=+0.00033)
  2018: 0.17810 / 0.17675  (Δ=-0.00136)
  2019: 0.14310 / 0.13951  (Δ=-0.00359)
  2021: 0.18148 / 0.17658  (Δ=-0.00490)
  2022: 0.19000 / 0.18566  (Δ=-0.00435)
  2023: 0.18510 / 0.18711  (Δ=+0.00201)
  2024: 0.15497 / 0.15579  (Δ=+0.00082)
  2025: 0.11983 / 0.12055  (Δ=+0.00072)


## 真实模拟历史回测（5 次独立运行）

每年独立模拟：若在该年参赛，只用该年之前的数据训练+校准，在该年上测试 Brier。
- 2021: 训练 2003–2020 → 测试 2021
- 2022: 训练 2003–2021 → 测试 2022
- 2023: 训练 2003–2022 → 测试 2023
- 2024: 训练 2003–2023 → 测试 2024
- 2025: 训练 2003–2024 → 测试 2025

每次运行：训练集内 LOSO 拟合 spline，再用全量训练集训练，预测该年。

下方运行 **Baseline（Elo + GLM）** 真实回测。

In [ ]:
# Baseline：Elo + GLM，无样本加权
import sys

sys.path.insert(0, "..")
from src.raddar_pipeline import run_realistic_backtest

run_realistic_backtest(
    data_dir="../march-machine-learning-mania-2026",
    test_years=[2021, 2022, 2023, 2024, 2025],
)

真实回测: 100%|██████████| 5/5 [02:44<00:00, 32.98s/it]


========== 真实模拟历史回测 (5 次独立运行) + 样本加权(指数衰减) ==========
每年：用该年之前的数据训练+校准，在该年上测试 Brier
  2021: 训练 2003..2020 → 测试 2021  Brier = 0.18163
  2022: 训练 2003..2021 → 测试 2022  Brier = 0.18784
  2023: 训练 2003..2022 → 测试 2023  Brier = 0.18516
  2024: 训练 2003..2023 → 测试 2024  Brier = 0.15559
  2025: 训练 2003..2024 → 测试 2025  Brier = 0.12122
  近5年平均 Brier: 0.16629
